In [34]:
import rasterio as rio
import numpy as np
from PIL import Image
from pathlib import Path
import logging
import detectree as dtr
import os
import pickle
from skimage.measure import find_contours
import tqdm
import geopandas as gpd
import logging
from shapely.ops import unary_union
from shapely.geometry import Polygon, MultiPolygon, GeometryCollection
from shapely.geometry import JOIN_STYLE

In [43]:
logger = logging.getLogger()
logger.setLevel(logging.ERROR)
# logger.addHandler(logging.StreamHandler(sys.stdout))

In [96]:
# Help funcs: 

def tif_to_png(tif_path, png_path):
    logger.info('Convert tif to png in processing...')
    with rio.open(tif_path) as src:
        array = src.read([1, 2, 3])  # Read the first three bands (R, G, B)
        rgb_array = np.dstack(array)  # Stack bands along the third dimension
        image = Image.fromarray(rgb_array, 'RGB')
        image.save(png_path)
    logger.info('Convert tif to png has been saved...')
    
    
def split_image(image_path, out_folder, size: int, skip_empty: bool = False):
    logger.info('Split images in processing...')

    if not Path(out_folder).exists():
        Path(out_folder).mkdir(parents=True, exist_ok=True)

    image_name = Path(image_path).name
    offsets = {}  # Dictionary to track offsets
    with Image.open(image_path) as img:
        width, height = img.size

        for i in range(0, width, size):
            for j in range(0, height, size):
                box = (i, j, i + size, j + size)
                crop = img.crop(box)
                if skip_empty and np.sum(np.array(crop)) == 0:
                    continue
                crop_filename = f"{IMAGE_NAME_PREFIX}_{i}_{j}.png"
                crop.save(os.path.join(out_folder, crop_filename))
                offsets[crop_filename] = (i, j)  # Track the offset
    logger.info(f'Images have been saved to {out_folder}')
    return offsets  # Return the offsets for further use


def create_polygons(mask):
    logger.info('Create polygons in processing...')
    simplification_tolerance = 1.0
    min_polygon_points = 3
    min_contour_points = 3
    join_mitre_leange = 1
    contours_level = 0.5
    min_area = 20.0

    contours = find_contours(mask, level=contours_level)
    polygons = []
    for contour in contours:
        if len(contour) >= min_contour_points:
            polygon = Polygon(contour[:, ::-1])
            if polygon.is_valid:
                polygons.append(polygon)

    buffered_polygons = [polygon.buffer(join_mitre_leange, join_style=JOIN_STYLE.mitre) for polygon in polygons]
    combined_polygons = unary_union(buffered_polygons)

    if isinstance(combined_polygons, MultiPolygon):
        combined_polygons = list(combined_polygons.geoms)
    elif isinstance(combined_polygons, Polygon):
        combined_polygons = [combined_polygons]
    else:
        combined_polygons = list(combined_polygons.geoms) if isinstance(combined_polygons, GeometryCollection) else []

    filtered_polygons = [polygon for polygon in combined_polygons
                         if len(polygon.exterior.coords) >= min_polygon_points and polygon.area >= min_area]

    simplified_polygons = [polygon.simplify(simplification_tolerance, preserve_topology=True) for polygon in filtered_polygons]

    final_polygons = []
    for polygon in simplified_polygons:
        if not polygon.is_empty and polygon.is_valid:
            final_polygons.append(polygon)

    logger.info('Create polygons has done.')
    return final_polygons
    # final_polygons now contains all the combined and simplified polygons, ensuring none are missing


    
def process_image(png_name):
    logger.info(f"Predicting model: {png_name}")
    # offset = offsets[png_name]

    clf = pickle.load(open('tree_model_new', 'rb'))
    clf_v1 = dtr.Classifier(clf=clf)

    # Predict models
    y_pred = clf_v1.predict_img(png_name)

    return y_pred

In [71]:
validation_image = 'img_0_1600.png.pickle'
prdict_image = 'images_split/img_0_1600.png'

In [72]:
validtion_metrix = pickle.load(open(validation_image, 'rb'))

In [91]:
pred_metrix = process_image(prdict_image)

/home/ec2-user/anaconda3/envs/python3/lib/python3.10/site-packages/rasterio/__init__.py:355: NotGeoreferencedWarning: Dataset has no geotransform, gcps, or rpcs. The identity matrix will be returned.
  dataset = DatasetReader(path, driver=driver, sharing=sharing, **kwargs)


In [92]:
from sklearn.metrics import jaccard_score

In [93]:
validtion_metrix = validtion_metrix == 255
pred_metrix = pred_metrix == 255

In [94]:
 jac = jaccard_score(pred_metrix, validtion_metrix, average='micro')

In [98]:
print("the score is:",jac)

the score is: 0.0
